### Attribute Lookup 🔍

In [2]:
# When you write:

# obj.name

# Python has to figure out:

# "Where is name?"

# It doesn't simply look inside obj.

In [3]:
# For example:

class A:
    x = 10

a = A()

print(a.x)

# a doesn't have x in its own __dict__.

# Yet:
# a.x

# returns:
# 10

# because Python searches the class after checking the instance.

10


In [4]:
# a.x
#  ↓
# a.__dict__?
#  ↓
# not found
#  ↓
# A.__dict__?
#  ↓
# found x = 10
#  ↓
# 10

In [5]:
# But the real lookup rules are more sophisticated, especially when descriptors and inheritance are involved.

### __getattribute__, __getattr__, and Descriptors

In [ ]:
# __getattribute__ vs __getattr__ 🔍

In [ ]:
# __getattribute__

# __getattribute__ is called for essentially every normal attribute access.

In [7]:
class Person:

    def __init__(self):
        self.name = "Shri"

    def __getattribute__(self, name):
        print("Accessing:", name)
        return super().__getattribute__(name)

In [8]:
# Now:

p = Person()

print(p.name)

Accessing: name
Shri


In [ ]:
# Output:

# Accessing: name
# Shri

# Because:

# p.name

# goes through:

# p.__getattribute__("name")

In [ ]:
# ⚠️ Why use super() here?

# Inside __getattribute__, if you do:

# return self.name

# you create a problem:

# self.name
#  ↓
# __getattribute__
#  ↓
# self.name
#  ↓
# __getattribute__
#  ↓
# self.name
#  ↓
# ...

# That's infinite recursion.

# So we normally use:
# super().__getattribute__(name)

# to perform the normal attribute lookup.

In [10]:
# __getattr__

# __getattr__ is different.
# It is called only when normal attribute lookup fails.
class Person:

    def __init__(self):
        self.name = "Shri"

    def __getattr__(self, name):
        return "Attribute not found"

In [11]:
p = Person()

print(p.name)
print(p.age)

Shri
Attribute not found


In [ ]:
# Output:

# Shri
# Attribute not found

# Why?

# p.name
# p.name
#  ↓
# normal lookup succeeds
#  ↓
# "Shri"

# __getattr__ isn't called.

# p.age
# p.age
#  ↓
# normal lookup fails
#  ↓
# __getattr__("age")
#  ↓
# "Attribute not found"

In [ ]:
# 🧠 The difference
# __getattribute__
# → called for attribute access

# __getattr__
# → called only when normal lookup fails

# Think of it like:

# obj.name
#    ↓
# __getattribute__("name")
#    ↓
# Found?
#  ┌───────┴───────┐
#  YES             NO
#  ↓                ↓
# return it     __getattr__("name")

# This distinction is very important.

### Descriptors — the engine behind property

In [12]:
# You've already used:

# @property
# def age(self):
#     ...

# A descriptor is the mechanism that allows Python to customize attribute access.

In [13]:
# A descriptor is an object that defines methods such as:

# __get__
# __set__
# __delete__

In [14]:
class Descriptor:

    def __get__(self, instance, owner):
        print("Getting value")
        return 100


In [15]:
# Then:

class Person:
    age = Descriptor()


p = Person()

print(p.age)

Getting value
100


In [ ]:
# Python sees:

# p.age

# and the descriptor can intercept that access:

# p.age
#  ↓
# Descriptor.__get__()
#  ↓
# 100

# So descriptors are one of the mechanisms behind powerful Python features such as:

# @property
# classmethod
# staticmethod

# and many frameworks use them heavily.

### Descriptors — __get__ 🔥

In [ ]:
# A descriptor is an object that controls how an attribute is accessed.

In [16]:
class Descriptor:

    def __get__(self, instance, owner):
        print("Getting value")
        return 100


class Person:
    age = Descriptor()


p = Person()

print(p.age)

Getting value
100


In [ ]:
# Output:

# Getting value
# 100

# What's happening?
# Normally you think:

# p.age

# means:
# Look for age inside p.

# But age isn't actually stored in p.__dict__.

In [ ]:
# Instead:

# Person
# └── age → Descriptor object

# When Python sees:

# p.age

# it recognizes that age is a descriptor and calls:

# Descriptor.__get__(p, Person)

# So:

# p.age
#  ↓
# Descriptor.__get__()
#  ↓
# 100

In [ ]:
# Understanding the two arguments

In [ ]:
# def __get__(self, instance, owner):

In [ ]:
# self
# The descriptor object itself.

# instance
# The object accessing the attribute.

In [ ]:
# Here:
# p.age

# means:
# instance → p

# owner
# The class containing the descriptor.

# owner → Person

In [2]:
# So conceptually:

# Descriptor.__get__(
#     descriptor,
#     p,
#     Person
# )

In [10]:
# Why is this useful?

# Remember your property example?

class Person:

    def __init__(self, age):
        self._age = age

    @property
    def age(self):
        return self._age

# When you write:

# p.age

# the property object is acting as a descriptor.

# So:

# @property
#    ↓
# descriptor mechanism
#    ↓
# __get__
#    ↓
# return _age

In [ ]:
# __set__ — Controlling Assignment

In [11]:
# We saw:

# p.age

# can trigger:

# __get__

# Similarly:

# p.age = 25

# can trigger:

# __set__

class AgeDescriptor:

    def __get__(self, instance, owner):
        return instance._age

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError("Age cannot be negative")

        instance._age = value


class Person:
    age = AgeDescriptor()

In [12]:
# Now:

p = Person()

p.age = 21
print(p.age)

21


In [13]:
# The assignment:

# p.age = 21

# essentially triggers:

# AgeDescriptor.__set__(p, 21)

# and stores:

# p._age = 21

# But:

# p.age = -5

# raises:

# ValueError: Age cannot be negative

In [ ]:
class Deceptor:
    
    def __get__(self,instance,owner):
        return instance._age
    
    def __set__(self,instance,value):
        if value < 0:
            raise ValueError("Invalid")
        
        instance._age = value
    

In [ ]:
class Person:
    age = Deceptor()
    
p = Person()
p.age = 21
print(p.age)

21


In [21]:
# 🧠 Descriptor pair
# p.age
#     ↓
# __get__()

# p.age = 21
#     ↓
# __set__()

In [ ]:
class Age:

    def __get__(self, instance, owner):
        return instance._age

    def __set__(self, instance, value):
        if value < 0:
            print("Invalid")
            return

        instance._age = value


class Person:
    age = Age()


p = Person()

p.age = 20
print(p.age)

p.age = -5
print(p.age)

In [22]:
# __delete__

In [24]:
# A descriptor can also control what happens when we do:

# del p.age

# using:

def __delete__(self, instance):
    ...

In [26]:
class Age:
    def __get__(self,instance,owner):
        return instance._age
    
    def __set__(self,instance,value):        
        instance._age = value
        
    def __delete__(self,instance):
        print("Deleting age")
        del instance._age
        
class Person:
    age = Age()
    
    
p = Person()
p.age = 21
print(p.age)

p.age = -5
print(p.age)

del p.age

print(p.age)

21
-5
Deleting age


AttributeError: 'Person' object has no attribute '_age'

In [ ]:
# One important descriptor concept

# You now know the three main descriptor operations:

# Operation	Descriptor method
# p.age	__get__
# p.age = 25	__set__
# del p.age	__delete__

# And because our descriptor has __set__, it is a data descriptor.

In [27]:
# Data vs Non-Data Descriptors

In [28]:
# 1. Data descriptor
# A descriptor that defines __set__ or __delete__.

class Data:
    def __get__(self, obj, owner):
        return "Descriptor"

    def __set__(self, obj, value):
        print("SET")


class A:
    x = Data()

a = A()
a.__dict__["x"] = "Instance"

print(a.x)

Descriptor


In [ ]:
# This is because a data descriptor gets priority over the instance's __dict__.

In [29]:
# 2. Non-data descriptor

# A descriptor with only __get__ is a non-data descriptor:

class NonData:
    def __get__(self, obj, owner):
        return "Descriptor"


class A:
    x = NonData()


a = A()
a.__dict__["x"] = "Instance"

print(a.x)

Instance


In [ ]:
# Data descriptor

# Has __set__ or __delete__.

# Data descriptor
#       ↓
# instance.__dict__

# So descriptor wins.

# Non-data descriptor

# Has only __get__.

# instance.__dict__
#       ↓
# Non-data descriptor

# So the instance value wins.

In [34]:
class check:
    def __init__(self,value):
        self.value = value
        
c = check(10)

c.__dict__

{'value': 10}

In [ ]:
# The complete lookup priority

# For a.x, roughly:

# 1. __getattribute__()
#         ↓
# 2. Data descriptor
#         ↓
# 3. a.__dict__
#         ↓
# 4. Non-data descriptor / class attribute
#         ↓
# 5. __getattr__()

In [ ]:
# One important distinction

# __dict__ stores instance attributes, not everything the object can access.

# For example:

# class Check:
#     species = "Human"

#     def __init__(self):
#         self.name = "Shri"

# c = Check()

# print(c.__dict__)
# print(c.species)

# Output:

# {'name': 'Shri'}
# Human

In [ ]:
# __set_name__ — making descriptors reusable

# Suppose we want a descriptor to validate different fields:

In [42]:
class Positive:
    def __set_name__(self,owner,name):
        self.name = name
        
    def __get__(self,instance,owner):
        return instance.__dict__[self.name]
    
    def __set__(self,instance,value):
        if value<0:
            raise ValueError("Invalid")
        instance.__dict__[self.name] = value
        
        
class Person:
    age =  Positive()
    salary = Positive()

In [36]:
p = Person()

p.age = 21
p.salary = 20000

print(p.age)
print(p.salary)

21
20000


In [ ]:
# When Python creates Person, it automatically calls:

# age_descriptor.__set_name__(Person, "age")
# salary_descriptor.__set_name__(Person, "salary")

# So each descriptor knows which attribute name it belongs to.

In [ ]:
# Person
#  ├── age    → Positive descriptor → name = "age"
#  └── salary → Positive descriptor → name = "salary"

In [40]:
# Then:

# p = Person()

# p.age = 22
# p.salary = 50000

# stores:

# p.__dict__
# # {'age': 22, 'salary': 50000}

In [ ]:
# Why is __set_name__ useful?

# Without it, we'd have to manually tell every descriptor its attribute name:

# age = Positive("age")
# salary = Positive("salary")

# With __set_name__:

# age = Positive()
# salary = Positive()

# Python tells the descriptor its name automatically.

In [43]:
class D:
    def __get__(self, obj, owner):
        print("GET")
        return self


class A:
    x = D()


a = A()

print(A.x)
print(a.x)

GET
GET


In [44]:
# A.x vs a.x

# When you do:

# A.x

# Python calls:

# D.__get__(None, A)

# Notice obj is None because there is no instance.

# When you do:

# a.x

# Python calls:

# D.__get__(a, A)
# So:

# A.x  → __get__(None, A)
# a.x  → __get__(a, A)

# That's why descriptors commonly do:

# def __get__(self, obj, owner):
#     if obj is None:
#         return self
#     return ...

# This allows:

# A.x    gives descriptor itself
# a.x    gives actual value